In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG        = "clutchlytics"
BRONZE_TABLE   = f"{CATALOG}.bronze.raw_nhl_game_summaries"
DIM_GAMES      = f"{CATALOG}.silver.dimGames"
DIM_TEAMS      = f"{CATALOG}.silver.dimTeams"
SILVER_TABLE   = f"{CATALOG}.silver.nhl_game_summaries"
 
SPORT  = "hockey"
LEAGUE = "nhl"
 
print(f"Source    : {BRONZE_TABLE}")
print(f"Target    : {SILVER_TABLE}")

In [0]:
# ── READ SOURCES ──────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from datetime import datetime, timezone
import json
 
bronze_df = spark.table(BRONZE_TABLE)
print(f"Bronze rows : {bronze_df.count()}")
 
# ── dimGames reference ──
dim_games = (
    spark.table(DIM_GAMES)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_game_id"),
        F.col("source_event_id").alias("dim_event_id"),
    )
)
 
# ── dimTeams reference ──
dim_teams = (
    spark.table(DIM_TEAMS)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_team_id"),
        F.col("team_id").cast("string").alias("dim_team_id"),
        F.col("abbreviation").alias("dim_abbreviation"),
    )
)
 
print(f"dimGames rows ({LEAGUE}) : {dim_games.count()}")
print(f"dimTeams rows ({LEAGUE}) : {dim_teams.count()}")

In [0]:
# ── PARSE TEAM STATS FROM BOXSCORE ───────────────────────────────────────────
# Expands Bronze (one row per game) into Silver (two rows per game — one per team).
 
ingested_at = datetime.now(timezone.utc).isoformat()
rows        = []
skipped     = []
 
bronze_rows = bronze_df.collect()
 
for br in bronze_rows:
    event_id    = br["event_id"]
    season      = br["season"]
    season_type = br["season_type"]
    round_num   = br["round"]
 
    try:
        boxscore = json.loads(br["boxscore_json"])
        teams    = boxscore.get("teams", [])
    except Exception as e:
        skipped.append((event_id, str(e)))
        print(f"  WARNING: Could not parse boxscore for event {event_id}: {e}")
        continue
 
    if len(teams) != 2:
        skipped.append((event_id, f"unexpected team count: {len(teams)}"))
        print(f"  WARNING: Expected 2 teams for event {event_id}, got {len(teams)}")
        continue
 
    def get_stat(team_block, stat_name, as_float=False):
        """Extract stat by name from team statistics array."""
        for stat in team_block.get("statistics", []):
            if stat.get("name") == stat_name:
                val = stat.get("displayValue")
                try:
                    return float(val) if as_float else int(float(val))
                except (TypeError, ValueError):
                    return None
        return None
 
    for team_block in teams:
        home_away    = team_block.get("homeAway")
        team_info    = team_block.get("team", {})
        team_id      = team_info.get("id")
        team_abbr    = team_info.get("abbreviation")
 
        rows.append({
            # ── Keys ──
            "source_event_id":    event_id,
            "team_id":            team_id,
            "home_away":          home_away,
 
            # ── Context ──
            "team_abbreviation":  team_abbr,
            "sport":              SPORT,
            "league":             LEAGUE,
            "season":             season,
            "season_type":        season_type,
            "round":              round_num,
 
            # ── Team stats ──
            "shots":              get_stat(team_block, "shotsTotal"),
            "hits":               get_stat(team_block, "hits"),
            "blocked_shots":      get_stat(team_block, "blockedShots"),
            "takeaways":          get_stat(team_block, "takeaways"),
            "giveaways":          get_stat(team_block, "giveaways"),
            "pp_goals":           get_stat(team_block, "powerPlayGoals"),
            "pp_opportunities":   get_stat(team_block, "powerPlayOpportunities"),
            "pp_pct":             get_stat(team_block, "powerPlayPct", as_float=True),
            "faceoffs_won":       get_stat(team_block, "faceoffsWon"),
            "faceoff_pct":        get_stat(team_block, "faceoffPercent", as_float=True),
            "penalties":          get_stat(team_block, "penalties"),
            "penalty_minutes":    get_stat(team_block, "penaltyMinutes"),
            "sh_goals":           get_stat(team_block, "shortHandedGoals"),
 
            # ── Metadata ──
            "ingested_at":        ingested_at,
            "source_table":       "bronze.raw_nhl_game_summaries",
        })
 
print(f"Rows built : {len(rows)}")
print(f"Skipped    : {len(skipped)}")
if skipped:
    for event_id, reason in skipped:
        print(f"  event {event_id}: {reason}")

In [0]:
# ── BUILD DATAFRAME + JOIN DIM REFERENCES ────────────────────────────────────
 
summaries_df = spark.createDataFrame(rows)
 
# ── Join dimGames → clutch_game_id ──
summaries_df = (
    summaries_df
    .join(
        dim_games,
        summaries_df.source_event_id == dim_games.dim_event_id,
        how="left"
    )
    .drop("dim_event_id")
)
 
# ── Join dimTeams → clutch_team_id ──
summaries_df = (
    summaries_df
    .join(
        dim_teams,
        summaries_df.team_id == dim_teams.dim_team_id,
        how="left"
    )
    .drop("dim_team_id", "dim_abbreviation")
)
 
# ── Warn on unmatched joins ──
unmatched_games = summaries_df.filter(F.col("clutch_game_id").isNull()).count()
unmatched_teams = summaries_df.filter(F.col("clutch_team_id").isNull()).count()
print(f"Unmatched dimGames : {unmatched_games} {'✓' if unmatched_games == 0 else '<-- investigate'}")
print(f"Unmatched dimTeams : {unmatched_teams} {'✓' if unmatched_teams == 0 else '<-- investigate'}")
 
# ── Final column order ──
summaries_df = summaries_df.select(
    "clutch_game_id",
    "clutch_team_id",
    "source_event_id",
    "team_id",
    "home_away",
    "team_abbreviation",
    "sport",
    "league",
    "season",
    "season_type",
    "round",
    "shots",
    "hits",
    "blocked_shots",
    "takeaways",
    "giveaways",
    "pp_goals",
    "pp_opportunities",
    "pp_pct",
    "faceoffs_won",
    "faceoff_pct",
    "penalties",
    "penalty_minutes",
    "sh_goals",
    "ingested_at",
    "source_table",
)
 
print(f"Total rows to write: {summaries_df.count()}")

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
# MERGE on source_event_id + team_id — safe for re-runs and new round uploads.
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if not table_exists:
    (
        summaries_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
else:
    summaries_df.createOrReplaceTempView("new_summaries")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_summaries AS source
        ON  target.source_event_id = source.source_event_id
        AND target.team_id         = source.team_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table: {SILVER_TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── Sample — first 10 rows ordered by game and team ──")
spark.sql(f"""
    SELECT
        clutch_game_id,
        source_event_id,
        team_abbreviation,
        home_away,
        shots,
        hits,
        blocked_shots,
        takeaways,
        giveaways,
        pp_goals,
        pp_opportunities,
        pp_pct,
        faceoffs_won,
        faceoff_pct,
        penalty_minutes,
        round
    FROM {SILVER_TABLE}
    ORDER BY clutch_game_id, home_away
    LIMIT 10
""").show(10, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_rows,
        COUNT(DISTINCT source_event_id)                        AS unique_games,
        COUNT(CASE WHEN clutch_game_id IS NULL  THEN 1 END)   AS null_clutch_game_ids,
        COUNT(CASE WHEN clutch_team_id IS NULL  THEN 1 END)   AS null_clutch_team_ids,
        COUNT(CASE WHEN shots IS NULL           THEN 1 END)   AS null_shots,
        COUNT(CASE WHEN hits IS NULL            THEN 1 END)   AS null_hits,
        COUNT(CASE WHEN home_away = 'home'      THEN 1 END)   AS home_rows,
        COUNT(CASE WHEN home_away = 'away'      THEN 1 END)   AS away_rows,
        ROUND(AVG(shots), 1)                                   AS avg_shots,
        ROUND(AVG(hits), 1)                                    AS avg_hits,
        ROUND(AVG(pp_pct), 1)                                  AS avg_pp_pct,
        SUM(pp_goals)                                          AS total_pp_goals,
        ROUND(AVG(faceoff_pct), 1)                            AS avg_faceoff_pct
    FROM {SILVER_TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)